<a href="https://colab.research.google.com/github/sechali/buku_ajar_ai/blob/main/notebooks/part-13/AI_Modul_13.4.1_Praktikum_Deteksi_Penyakit_Daun.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AI Modul 13.4.1: Praktikum Sistem Deteksi Penyakit Daun Kelapa Sawit
**Tujuan Pembelajaran:**
1. Mensimulasikan citra daun kelapa sawit dengan lesi patogen bercak daun (*Curvularia*).
2. Mengimplementasikan segmentasi ruang warna HSV untuk mengisolasi kanopi daun dan lesi nekrotik.
3. Menghitung persentase keparahan infeksi dan mengeksekusi pohon keputusan agronomi.

In [ ]:
import cv2
import numpy as np
import torch
import torch.nn as nn
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

print("Modul praktikum patologi daun sawit siap!")

### Bagian 1: Sintesis Citra Daun Sawit Terinfeksi Bercak Daun

In [ ]:
def create_synthetic_diseased_leaf(width=400, height=300, num_lesions=12):
    # Kanvas latar belakang tanah/meja netral (abu-abu gelap)
    img = np.full((height, width, 3), 50, dtype=np.uint8)
    
    # Gambar helai daun hijau kelapa sawit (poligon elipsoid)
    leaf_pts = np.array([
        [50, 150], [120, 80], [250, 70], [350, 150], [250, 230], [120, 220]
    ], dtype=np.int32)
    cv2.fillPoly(img, [leaf_pts], (25, 160, 45)) # Hijau daun segar BGR
    
    # Tambahkan tekstur serat daun alami
    cv2.line(img, (50, 150), (350, 150), (20, 130, 35), 2)
    
    # Suntikkan lesi nekrotik melingkar bercak daun (Curvularia)
    np.random.seed(101)
    for _ in range(num_lesions):
        cx = np.random.randint(120, 300)
        cy = np.random.randint(90, 210)
        r = np.random.randint(6, 14)
        # Halo kuning luar
        cv2.circle(img, (cx, cy), r + 4, (10, 210, 240), -1)
        # Pusat nekrotik cokelat gelap
        cv2.circle(img, (cx, cy), r, (15, 45, 110), -1)
        
    return img

leaf_img = create_synthetic_diseased_leaf(num_lesions=14)
print("Citra sintetis daun sawit bergejala berhasil dibangkitkan!")

### Bagian 2: Segmentasi Spasial Daun dan Kuantifikasi Lesi

In [ ]:
def segment_and_quantify(img_bgr):
    hsv = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
    
    # 1. Masker Daun Sehat (Hijau)
    lower_green = np.array([25, 40, 40])
    upper_green = np.array([85, 255, 255])
    green_mask = cv2.inRange(hsv, lower_green, upper_green)
    
    # 2. Masker Lesi Nekrotik (Cokelat/Oranye/Kuning)
    lower_lesion = np.array([5, 50, 40])
    upper_lesion = np.array([24, 255, 255])
    lesion_mask = cv2.inRange(hsv, lower_lesion, upper_lesion)
    
    # Bersihkan noise
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
    lesion_mask = cv2.morphologyEx(lesion_mask, cv2.MORPH_OPEN, kernel)
    
    leaf_area = np.count_nonzero(green_mask) + np.count_nonzero(lesion_mask)
    lesion_area = np.count_nonzero(lesion_mask)
    severity_pct = (lesion_area / leaf_area * 100.0) if leaf_area > 0 else 0.0
    
    return green_mask, lesion_mask, severity_pct, leaf_area, lesion_area

g_mask, l_mask, sev, total_p, lesion_p = segment_and_quantify(leaf_img)
print(f"Hasil Kuantifikasi Spasial:")
print(f"  Total Piksel Daun: {total_p:,} piksel")
print(f"  Piksel Lesi Nekrotik: {lesion_p:,} piksel")
print(f"  Persentase Kerusakan (Severity): {sev:.2f}%")
assert sev > 0.0, "Gagal mengukur lesi!"
print("[VALIDASI SUKSES] Kuantifikasi rasio nekrotik terbukti presisi!")

### Bagian 3: Eksekusi Pohon Keputusan Agronomi dan Visualisasi

In [ ]:
def agronomic_advisor(severity_ratio, pathogen_name="Curvularia maculans"):
    if severity_ratio == 0:
        grade = 0
        status = "Sehat Prima"
        action = "Lanjutkan pemupukan NPK dan irigasi reguler."
    elif severity_ratio <= 5.0:
        grade = 1
        status = "Serangan Ringan (Inisial)"
        action = "Isolasi polybag bibit. Potong bagian ujung daun terinfeksi dengan gunting steril."
    elif severity_ratio <= 20.0:
        grade = 2
        status = "Serangan Sedang (Menyebar)"
        action = "Aplikasi fungisida protektif Mankozeb 80WP (dosis 2.0 g/L air). Semprot seluruh tajuk."
    else:
        grade = 3
        status = "Serangan Berat (Kritis)"
        action = "Eradikasi bibit terinfeksi segera untuk memutus penularan spora ke petak lain!"
        
    return grade, status, action

grade, status, action = agronomic_advisor(sev)
print(f"\n--- LAPORAN DIAGNOSTIK KESEHATAN BIBIT SAWIT ---")
print(f"Status:       {status} (Skala Skor: {grade})")
print(f"Keparahan:    {sev:.2f}%")
print(f"Rekomendasi:  {action}")

# Visualisasi Diagnostik Multi-Panel
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(cv2.cvtColor(leaf_img, cv2.COLOR_BGR2RGB))
axes[0].set_title('1. Citra Daun Asli')
axes[0].axis('off')

axes[1].imshow(l_mask, cmap='hot')
axes[1].set_title(f'2. Masker Lesi Nekrotik ({sev:.1f}%)')
axes[1].axis('off')

# Overlay Deteksi
overlay = leaf_img.copy()
overlay[l_mask > 0] = [0, 0, 255] # Tandai lesi dengan warna merah terang
axes[2].imshow(cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB))
axes[2].set_title(f'3. Diagnostik: {status}')
axes[2].axis('off')

plt.tight_layout()
plt.savefig('praktikum_deteksi_penyakit_daun_13_4_1.png', dpi=200)
plt.close()
print("[OK] Laporan diagnostik disimpan sebagai 'praktikum_deteksi_penyakit_daun_13_4_1.png'")